## About:

### This notebook contains data generation pipeline using huggingface models using AutoModelForCausalLM along with RAG. The questions are picked from LSST forum page. Multiple models can be used just by adding its huggingface identifier in the list named model_list below.

In [7]:
!pip install langchain langchain-community openai transformers sentence-transformers pandas langchain_qdrant langchain_huggingface


In [8]:
import textwrap
from uuid import uuid4
from pathlib import Path
import warnings
import pandas as pd

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import LlamaCpp
from langchain_qdrant import Qdrant
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient

warnings.filterwarnings("ignore")


In [ ]:
qdrant_path       = Path("resources/rubin_qdrant")
qdrant_collection = "rubin_telescope"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

client = QdrantClient(path=str(qdrant_path))
db = Qdrant(
    client           = client,
    collection_name  = qdrant_collection,
    embeddings       = embedding_model,
)

retriever = db.as_retriever(
    search_type = "mmr",
    search_kwargs= {"k": 2},
)


In [10]:
def load_hf_llm(model_name: str):
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model     = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code = True,
    )

    text_gen = pipeline(
        "text-generation",
        model      = model,
        tokenizer  = tokenizer,
        max_new_tokens = 256,
        temperature    = 0.7,
        device         = 0 if torch.cuda.is_available() else -1,
    )

    return HuggingFacePipeline(pipeline=text_gen)


In [11]:
prompt_template = PromptTemplate.from_template(
    textwrap.dedent(
        """\
        You are an astrophysics expert. Please answer the question on astrophysics based on the following context:

        {context}

        Question: {question}
        """
    )
)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

def build_chain(model_name: str):
    hf_llm = load_hf_llm(model_name)

    return (
        {
            "context" : retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt_template
        | hf_llm
    )


In [ ]:

excel_path = "data/lsst_forum_responses_5.xlsx"
df_q = pd.read_excel(excel_path)
questions  = df_q["question"].dropna().tolist()

# Add as many HF models as you like here
model_list = [
    "allenai/OLMo-1B",
    "allenai/OLMo-1B-0724-hf",
    "allenai/OLMo-2-0425-1B-Instruct"
]


all_results = {"question": questions}

for model_name in model_list:
    print(f"\n↪ Generating answers with: {model_name}")
    chain = build_chain(model_name)

    answers = []
    for q in questions:
        ans = chain.invoke(q).strip()
        answers.append(ans)

    col = model_name.split("/")[-1]      # nicify column name
    all_results[col] = answers


out_df = pd.DataFrame(all_results)
out_df.to_csv("rag_hf_qdrant_responses.csv", index=False)
out_df.head()



↪ Generating answers with: allenai/OLMo-1B


Device set to use cpu



↪ Generating answers with: allenai/OLMo-1B-0724-hf


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 59.94it/s]
Device set to use cpu



↪ Generating answers with: allenai/OLMo-2-0425-1B-Instruct


Device set to use cpu


,question,OLMo-1B,OLMo-1B-0724-hf,OLMo-2-0425-1B-Instruct
0,"Hi, \nI’m following this tutorial: The LSST S...",You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...
1,I have the following C++ class : \n class CcdI...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...
2,Question on how forced photometry will be run ...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...
3,"Hi there, \n Is there some way I find out what...",You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...
4,I’m having trouble building FFTW with texinfo ...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...,You are an astrophysics expert. Please answer ...
